# Qwen-TTS 입문 매뉴얼
부제: 구글 코랩에서 설치하고, CustomVoice부터 사용해 보는 실습 교재

---

## 1. Qwen-TTS란?

Qwen-TTS는 Qwen 팀이 공개한 오픈소스 텍스트-투-스피치(TTS) 모델 시리즈다.
문장을 음성으로 바꾸는 기능뿐 아니라, 다음과 같은 기능을 지원한다.

- 다국어 음성 합성
- 자연어 지시를 통한 스타일 제어
- 스트리밍 기반 저지연 합성
- 참조 음성을 이용한 보이스 클로닝
- 미리 준비된 화자를 사용하는 TTS

공식 자료 기준으로 Qwen3-TTS는 다음 10개 언어를 지원한다.

- Chinese
- English
- Japanese
- Korean
- German
- French
- Russian
- Portuguese
- Spanish
- Italian


## 8. Base 모델 사용 방법

Base 모델은 참조 음성을 넣어서 보이스 클로닝을 할 때 사용한다.

### 8.1 기본 개념
아래 두 입력이 필요하다.

- `ref_audio`: 참조 음성 파일
- `ref_text`: 참조 음성에 실제로 들어 있는 텍스트

즉, 참조 음성을 모델에 주고, 그 음성의 특징을 따라 새 문장을 읽도록 하는 방식이다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U qwen-tts soundfile
!apt-get -y install ffmpeg

2. 모델 로드

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("DEVICE:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map=DEVICE,
    dtype=DTYPE,
)

3. 참조 텍스트 입력 + 파일 업로드 + 클론 생성 + TTS 오디오 생성

In [ ]:
import os
import subprocess
import soundfile as sf
from google.colab import files
from IPython.display import Audio, display

print("=== Qwen-TTS Voice Clone: 파일 업로드 방식 ===")

# 1. 참조 문장 입력
ref_text = input("\n[1단계] 참조 음성 파일에 실제로 들어 있는 문장을 입력하세요:\n> ").strip()
#ref_text = "동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라만세.무궁화 삼천리 화려강산 대한사람 대한으로 길이 보전하세"
if not ref_text:
    raise ValueError("참조 음성 문장을 입력해야 합니다.")

# 2. 파일 업로절
print("\n[2단계] 참조 음성 파일을 업로드하세요. (wav/mp3/m4a/webm 등)")
#uploaded = files.upload(절

#if not uploaded:
#    raise ValueError("업로드된 파일이 없습니다.")

#uploaded_name = next(iter(uploaded.keys()))
#src_path = f"/content/{uploaded_name}"
src_path =  f"/content/drive/MyDrive/2026/clone_voice/recored_voice/김남이.m4a"
wav_path = "/content/drive/MyDrive/2026/clone_voice/recored_voice/김남이.wav"

# 3. wav 변환
subprocess.run([
    "ffmpeg", "-y",
    "-i", src_path,
    "-ac", "1",
    "-ar", "16000",
    wav_path
], check=True)

# 4. 길이 확인
audio, sr = sf.read(wav_path)
duration = len(audio) / sr
print(f"\n참조 음성 길이: {duration:.2f}초")

if duration < 3.0:
    print("안내: 참조 음성은 3초 이상을 권장합니다.")
else:
    print("참조 음성 길이가 적절합니다.")

# 5. 준비 완료 안내
print("\n[3단계] 목소리 참조 파일이 정상적으로 준비되었습니다.")
print("이제 이 목소리로 읽을 문장을 입력하면 음성을 생성합니다.")

# 6. 생성 문장 입력
target_text = input("\n[4단계] 생성할 음성 스크립트를 입력하세요:\n> ").strip()
if not target_text:
    raise ValueError("생성할 문장을 입력해야 합니다.")
#target_text = "내 이름은 김남이입니다. 만나서 반갑습니다."
print(target_text)

# 7. 음성 생성
wavs, out_sr = model.generate_voice_clone(
    text=target_text,
    language="Korean",
    ref_audio=wav_path,
    ref_text=ref_text,
)

# 8. 저장 및 재생
out_path = "/content/drive/MyDrive/2026/clone_voice/recored_voice/voice_clone_my_name_is_kimname.wav"
sf.write(out_path, wavs[0], out_sr)

print("\n음성 생성이 완료되었습니다.")
print("저장 위치:", out_path)
display(Audio(out_path))


#ref_text = "동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라만세.무궁화 삼천리 화려강산 대한사람 대한으로 길이 보전하세"
#target_text = "내 이름은 김남이입니다. 만나서 반갑습니다. 우리 모두 좋은 세상에에서 행복하게 살아요."

#ref_text = "부동산 카페에 그 중국인 1개월 내고 119개월 그 한 번에 납입하고 그거 중국 가서 연금 받는다고. 그것 때문에 지금 정부에서"
#target_text = "내 이름은 한미란입니다 만나서 반갑습니다 우리 모두 좋은 세상에서 행복하게 살아요 "

#ref_text = "어 야 아빠가 없으면은 공부 더 열심히 할 거 아니야. 어? 임마이"
#target_text = "내 이름은 김남이입니다. 만나서 반갑습니다. 우리 모두 좋은 세상에에서 행복하게 살아요."
